# 01 — Data Cleaning & Standardization

**Objective:** Ingest the raw FCC complaint export, standardize column names and data types, normalize free-text fields, and produce a clean intermediate dataset ready for classification.

**Why this matters:** The raw FCC file uses inconsistent casing, mixed date formats, and column names that don't translate cleanly to a warehouse schema. Every downstream query and model depends on this step being repeatable and explicit. Decisions made here (e.g., how we handle null states, how we parse dates) should be documented so a reviewer can audit the lineage.

**Input:** `data/raw/comcast_fcc_complaints_2015.csv`  
**Output:** `data/processed/cleaned_comcast_complaints.csv`

In [ ]:
import pandas as pd

RAW_PATH = "../data/raw/comcast_fcc_complaints_2015.csv"
PROCESSED_PATH = "../data/processed/cleaned_comcast_complaints.csv"

## 1. Load & Inspect Raw Data

In [ ]:
df = pd.read_csv(RAW_PATH)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head(3)

In [ ]:
# Null counts before cleaning — document what we're starting with
df.isnull().sum()

## 2. Rename Columns

Standardizing to snake_case. The `Description` field contains the full complaint narrative and is more useful than the short `Customer Complaint` title — we replace `complaint_text` with the description content.

In [ ]:
df.columns = df.columns.str.strip()

df = df.rename(columns={
    "Ticket #": "ticket_id",
    "Customer Complaint": "complaint_title",
    "Description": "complaint_text",
    "Date": "date",
    "Time": "time",
    "Received Via": "channel",
    "City": "city",
    "State": "state",
    "Zip code": "zip_code",
    "Status": "status",
    "Filing on Behalf of Someone": "on_behalf",
})

df.columns.tolist()

## 3. Normalize Text & Categorical Fields

In [ ]:
# Lowercase and strip whitespace from free-text and low-cardinality fields
df["complaint_text"] = (
    df["complaint_text"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\n", " ", regex=True)
)

df["status"] = df["status"].astype(str).str.lower().str.strip()
df["channel"] = df["channel"].astype(str).str.lower().str.strip()

# Normalize state — strip whitespace, title-case for consistency
# Note: raw data contains both 'District Of Columbia' and 'District of Columbia'
df["state"] = df["state"].astype(str).str.strip().str.title()

print("Status values:", df["status"].unique())
print("Channel values:", df["channel"].unique())

## 4. Parse Dates & Extract Time Features

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.day_name()

# Flag any rows where date parsing failed
null_dates = df["date"].isnull().sum()
print(f"Rows with unparseable dates: {null_dates}")

## 5. Drop the Short Title Field

`complaint_title` (original: `Customer Complaint`) is a brief user-supplied summary with no analytical value — full narrative is in `complaint_text`. Dropping it keeps the schema clean.

In [ ]:
df = df.drop(columns=["complaint_title"], errors="ignore")
df.dtypes

## 6. Final Quality Check

In [ ]:
print(f"Final row count: {len(df)}")
print(f"\nNull counts after cleaning:")
print(df.isnull().sum())
print(f"\nDuplicate ticket IDs: {df['ticket_id'].duplicated().sum()}")
print(f"\nDate range: {df['date'].min().date()} to {df['date'].max().date()}")

## 7. Export Processed Dataset

In [ ]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved to {PROCESSED_PATH}")